# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gaganakhil21/flyrank-1/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I'm choosing Lane 4 — CTR / Engagement Opportunity Scoring.
It targets the cheapest fixable gap in search: pages that already win impressions but lose the
click, where a title/meta fix takes hours, not a full rewrite. The starter pipeline uses CTR as
one feature among ~26 feeding a decline label; making it the target turns a buried signal into
the deliverable. It also has the deepest data support (impressions are the densest signal in the
release) and a transparent baseline to beat (expected CTR by position tier), so a model only
earns its place by beating that baseline — decision-support, not just a model.


In [7]:
from pathlib import Path
import subprocess, sys

NAME = "content_refresh_anonymized.csv"
candidates = [
    Path("data/raw") / NAME,          # repo root
    Path("../data/raw") / NAME,       # work/
    Path("../../data/raw") / NAME,    # work/notebooks/
]
path = next((p for p in candidates if p.exists()), None)

if path is None and "google.colab" in sys.modules:
    # Colab only downloads the notebook, not the data — clone the repo
    repo = Path("/content/flyrank_ml")
    if not (repo / "data" / "raw" / NAME).exists():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/ashoktanakanti/flyrank_ml.git", str(repo)],
            check=True,
        )
    path = repo / "data" / "raw" / NAME

if path is None:
    raise FileNotFoundError(
        "CSV not found — on your own machine run this notebook from inside your repo clone."
    )

import pandas as pd
df = pd.read_csv(path)
print(f"Loaded: {len(df):,} rows x {df.shape[1]} cols from {path}")

Loaded: 30,000 rows x 44 cols from /content/flyrank_ml/data/raw/content_refresh_anonymized.csv


## 2. The question: decision, action, cost of a wrong call

Decision improved: where a content editor spends a fixed weekly review budget.

Who acts: the
content/SEO editor, using the ranked queue + reason codes as a starting point.

Action: open the top-ranked page, read its reason code, then rewrite title/meta, fix intent match, improve the snippet, or explicitly monitor.

Cost of a wrong call: a false positive wastes one review slot
(cheap, self-correcting); a false negative means a page keeps winning impressions and losing clicks week after week, invisibly — the expensive direction. Lane-specific trap: top-ranked pages often have tiny impression counts where CTR is mostly noise, so minimum-volume gating is part of the scoring design, not cleanup.

In [8]:
print("Provisional lane: CTR / Engagement Opportunity Scoring (Lane 4), until end of Week 4")
print("Decision: where a fixed weekly review budget goes | Action: editor fixes title/meta/intent or monitors, from reason codes")
print("Cost of wrong call: false negative (visible page keeps losing clicks unnoticed) >> false positive (one wasted review slot)")
print("This is decision-support: transparent baseline -> ranked queue -> client-holdout validation -> reason codes. Not just a model.")

Provisional lane: CTR / Engagement Opportunity Scoring (Lane 4), until end of Week 4
Decision: where a fixed weekly review budget goes | Action: editor fixes title/meta/intent or monitors, from reason codes
Cost of wrong call: false negative (visible page keeps losing clicks unnoticed) >> false positive (one wasted review slot)
This is decision-support: transparent baseline -> ranked queue -> client-holdout validation -> reason codes. Not just a model.


## 3. Quick look at the data (2-3 real numbers)

I loaded the starter CSV directly and pulled three numbers that argue this lane is worth the next 7 weeks.

In [9]:
n = len(df)
pool = df[(df["impressions_90d"] >= 1000) & (df["avg_position"] > 0)
          & (df["avg_position"] <= 20) & (df["ctr"] < 1.0)]
print(f"Number 1 - visible, well-positioned, under-capturing pages: {len(pool):,} of {n:,} ({len(pool)/n:.1%})")
TIER_ORDER = ["top_3", "page_1", "striking", "page_3_5", "deep"]
tier_ctr = (df[df["avg_position"] > 0].groupby("position_tier")["ctr"].median()
            .reindex(TIER_ORDER).dropna().round(2))
print("Number 2 - median CTR (x100) by position tier (position matters, so adjust for it):")
print(tier_ctr.to_string())
top3 = df[df["position_tier"] == "top_3"]
print(f"Number 3 - top_3 pages: median impressions/90d = {top3['impressions_90d'].median():,.0f} "
      f"(n={len(top3):,}) - tiny samples, so minimum-volume gating comes first")

Number 1 - visible, well-positioned, under-capturing pages: 9,553 of 30,000 (31.8%)
Number 2 - median CTR (x100) by position tier (position matters, so adjust for it):
position_tier
top_3       0.00
page_1      0.16
striking    0.11
page_3_5    0.03
deep        0.00
Number 3 - top_3 pages: median impressions/90d = 3 (n=2,321) - tiny samples, so minimum-volume gating comes first


## 4. Careful words: what I can and can't claim

What I can claim: that in this 90-day anonymized slice, pages with these measured signals tended
to capture fewer clicks than comparable pages at similar positions; that the output is a ranked,
directional review queue for a human editor, with reason codes they can always inspect; every
number is an observed GSC/GA4 aggregate from the starter data, not a theory about how search
works.

What I can never claim: that fixing a title or meta CAUSES CTR to rise (that would need an
experiment, not this data); that low CTR always means bad metadata (it can be intent mismatch,
SERP features, or small-sample noise — the top_3 median of ~53 impressions shows how noisy
small samples are); anything about "predicting Google's algorithm"; or that results generalize
beyond this 30,000-page / 32-client snapshot until re-earned on the full warehouse with proper
validation. the data dictionary quotes ~53 for this median; my live compute on this slice says 3 — I trust my computed output over the doc, which is the careful-words habit in action.

In [10]:
can_say = ["observed", "measured", "directional", "decision-support"]
cannot_say = ["causal proof", "predicting Google", "guaranteed lift"]
print("I can say:  ", ", ".join(can_say))
print("I can't say:", ", ".join(cannot_say))

I can say:   observed, measured, directional, decision-support
I can't say: causal proof, predicting Google, guaranteed lift


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.